# 00 — Structured Data — All in One
Exécute ce notebook si tu veux refaire tout le pipeline dans un seul fichier.

**Avant exécution : modifie `src/config.py`.**


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from config import DATA_DIR, OUTPUT_DIR, FILES, TARGET_RMPM_IDS
from structured_utils import *

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 200)


## 1. Chargement


In [ ]:
datasets = {}
for key, filename in FILES.items():
    path = DATA_DIR / filename
    print(f"{key}: {path} | exists={path.exists()}")
    datasets[key] = load_excel(path)
    print("shape:", datasets[key].shape)


In [ ]:
for name, df in datasets.items():
    print("\n" + "="*100)
    print(name.upper(), df.shape)
    print("="*100)
    display(df.head())
    display(pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing_n": df.isna().sum(),
        "missing_pct": (df.isna().mean()*100).round(2)
    }).sort_values("missing_pct", ascending=False))
    print("duplicates:", df.duplicated().sum())
    if "RMPM_ID" in df.columns:
        print("RMPM_ID uniques:", df["RMPM_ID"].nunique())


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for name, df in datasets.items():
    df.to_pickle(OUTPUT_DIR / f"00_raw_{name}.pkl")
print("Raw snapshots saved.")


## 2. Nettoyage / préparation


In [ ]:
rmpm = pd.read_pickle(OUTPUT_DIR / "00_raw_rmpm.pkl")
legal_entity = pd.read_pickle(OUTPUT_DIR / "00_raw_legal_entity.pkl")
facility = pd.read_pickle(OUTPUT_DIR / "00_raw_facility.pkl")
covenant = pd.read_pickle(OUTPUT_DIR / "00_raw_covenant.pkl")
asset = pd.read_pickle(OUTPUT_DIR / "00_raw_asset.pkl")


In [ ]:
for df in [rmpm, legal_entity, facility, covenant, asset]:
    if "RMPM_ID" in df.columns:
        df["RMPM_ID"] = clean_identifier(df["RMPM_ID"])
    if "FACILITY_UNIQUE_ID" in df.columns:
        df["FACILITY_UNIQUE_ID"] = clean_identifier(df["FACILITY_UNIQUE_ID"])

rmpm = filter_target_counterparties(rmpm, TARGET_RMPM_IDS)
legal_entity = filter_target_counterparties(legal_entity, TARGET_RMPM_IDS)
facility = filter_target_counterparties(facility, TARGET_RMPM_IDS)
covenant = filter_target_counterparties(covenant, TARGET_RMPM_IDS)
asset = filter_target_counterparties(asset, TARGET_RMPM_IDS)

rmpm = convert_date_columns(rmpm)
legal_entity = convert_date_columns(legal_entity)
facility = convert_date_columns(facility)
covenant = convert_date_columns(covenant)
asset = convert_date_columns(asset)

legal_entity = normalize_boolean_columns(legal_entity)
facility = normalize_boolean_columns(facility)
covenant = normalize_boolean_columns(covenant)


In [ ]:
for c in ["PD_CLEAN","GRR_SU"]:
    if c in rmpm.columns:
        rmpm[c] = rmpm[c].map(parse_number)

for c in ["FINAL_TAKE_AMOUNT","OUTSTANDING_GLOBAL","NET_AMOUNT","GROSS_AMOUNT","TENOR","TENOR_YEARS"]:
    if c in facility.columns:
        facility[c] = facility[c].map(parse_number)

for c in ["SAF","TOTAL_SAF"]:
    if c in legal_entity.columns:
        legal_entity[c] = legal_entity[c].map(parse_number)


In [ ]:
if "SITUATION_DATE" in rmpm.columns:
    rmpm = rmpm.sort_values(["RMPM_ID","SITUATION_DATE"]).reset_index(drop=True)
if "DATE_OF_COMMITTEE" in legal_entity.columns:
    legal_entity = legal_entity.sort_values(["RMPM_ID","DATE_OF_COMMITTEE"]).reset_index(drop=True)
facility_sort = [c for c in ["RMPM_ID","FACILITY_UNIQUE_ID","DATE_OF_COMMITTEE"] if c in facility.columns]
if facility_sort:
    facility = facility.sort_values(facility_sort).reset_index(drop=True)

kyc_cols = [c for c in legal_entity.columns if "KYC" in canonical_column_name(c)]
if kyc_cols:
    legal_entity[kyc_cols] = legal_entity.groupby("RMPM_ID", group_keys=False)[kyc_cols].ffill()

rmpm_rating_col = find_column(rmpm, "Counterparty Rating", "COUNTERPARTY_RATING")
if rmpm_rating_col:
    rmpm["RATING_NUM"] = rmpm[rmpm_rating_col].map(parse_rating)
legal_rating_col = find_column(legal_entity, "COUNTERPARTY_RATING", "Counterparty Rating")
if legal_rating_col:
    legal_entity["RATING_NUM"] = legal_entity[legal_rating_col].map(parse_rating)


In [ ]:
for name, df in {
    "rmpm":rmpm, "legal_entity":legal_entity, "facility":facility,
    "covenant":covenant, "asset":asset
}.items():
    df.to_pickle(OUTPUT_DIR / f"01_clean_{name}.pkl")
    print(name, df.shape)


## 3. Coverage / qualité


In [ ]:
rmpm = pd.read_pickle(OUTPUT_DIR / "01_clean_rmpm.pkl")
legal_entity = pd.read_pickle(OUTPUT_DIR / "01_clean_legal_entity.pkl")
facility = pd.read_pickle(OUTPUT_DIR / "01_clean_facility.pkl")
covenant = pd.read_pickle(OUTPUT_DIR / "01_clean_covenant.pkl")
asset = pd.read_pickle(OUTPUT_DIR / "01_clean_asset.pkl")


In [ ]:
coverage_df = build_coverage(rmpm, legal_entity, facility, covenant, asset, TARGET_RMPM_IDS)
display(coverage_df)
coverage_df.to_csv(OUTPUT_DIR/"coverage.csv", index=False)


In [ ]:
for name, df in {"RMPM":rmpm,"LEGAL":legal_entity,"FACILITY":facility,"COVENANT":covenant,"ASSET":asset}.items():
    print("\n", name)
    print("rows:", len(df))
    print("duplicates:", df.duplicated().sum())
    print("missing RMPM_ID:", df["RMPM_ID"].isna().sum() if "RMPM_ID" in df.columns else "N/A")


## 4. RMPM


In [ ]:
rmpm = pd.read_pickle(OUTPUT_DIR / "01_clean_rmpm.pkl")


In [ ]:
for column, delta in [("PD_CLEAN","DELTA_PD"),("GRR_SU","DELTA_GRR"),("RATING_NUM","DELTA_RATING")]:
    if column in rmpm.columns:
        rmpm[delta] = rmpm.groupby("RMPM_ID")[column].diff()
display(rmpm.head(20))


In [ ]:
def plot_metric(cp, metric, ylabel=None):
    data = rmpm[rmpm["RMPM_ID"] == cp].sort_values("SITUATION_DATE")
    if metric not in data.columns:
        print("Missing:", metric); return
    plt.figure(figsize=(14,5))
    plt.plot(data["SITUATION_DATE"], data[metric], marker="o")
    plt.title(f"{metric} trajectory - {cp}")
    plt.xlabel("Date"); plt.ylabel(ylabel or metric); plt.grid(alpha=.3)
    plt.xticks(rotation=45); plt.tight_layout(); plt.show()

for cp in TARGET_RMPM_IDS:
    plot_metric(cp, "PD_CLEAN", "PD")
    plot_metric(cp, "GRR_SU", "GRR")
    plot_metric(cp, "RATING_NUM", "Rating")


In [ ]:
rmpm.to_pickle(OUTPUT_DIR/"02_rmpm_enriched.pkl")


## 5. Legal Entity


In [ ]:
legal_entity = pd.read_pickle(OUTPUT_DIR / "01_clean_legal_entity.pkl")


In [ ]:
variables = [
    "COUNTERPARTY_RATING","INTRINSIC_RATING","KYC","WATCHLIST_INDICATOR",
    "UTP_TEST","CONFIRMED_UTP","NPE","Close to Default","SAF","TOTAL_SAF"
]
present = [c for c in variables if c in legal_entity.columns]
print("Variables present:", present)
legal_changes = extract_changes(legal_entity, present)
display(legal_changes)
legal_changes.to_csv(OUTPUT_DIR/"legal_entity_changes.csv", index=False)


## 6. Facilities


In [ ]:
facility = pd.read_pickle(OUTPUT_DIR / "01_clean_facility.pkl")


In [ ]:
if "OUTSTANDING_GLOBAL" in facility.columns and "FINAL_TAKE_AMOUNT" in facility.columns:
    facility["UTILIZATION"] = facility["OUTSTANDING_GLOBAL"] / facility["FINAL_TAKE_AMOUNT"]
    facility.loc[facility["FINAL_TAKE_AMOUNT"] == 0, "UTILIZATION"] = np.nan

agg = {"FACILITY_UNIQUE_ID":"nunique"}
if "FINAL_TAKE_AMOUNT" in facility.columns: agg["FINAL_TAKE_AMOUNT"] = "sum"
if "OUTSTANDING_GLOBAL" in facility.columns: agg["OUTSTANDING_GLOBAL"] = "sum"

facility_summary = facility.groupby(["RMPM_ID","DATE_OF_COMMITTEE"]).agg(agg).reset_index()
facility_summary = facility_summary.rename(columns={
    "FACILITY_UNIQUE_ID":"NB_FACILITIES",
    "FINAL_TAKE_AMOUNT":"TOTAL_FINAL_TAKE",
    "OUTSTANDING_GLOBAL":"TOTAL_OUTSTANDING"
})
if "TOTAL_FINAL_TAKE" in facility_summary.columns and "TOTAL_OUTSTANDING" in facility_summary.columns:
    facility_summary["UTILIZATION"] = facility_summary["TOTAL_OUTSTANDING"] / facility_summary["TOTAL_FINAL_TAKE"]
display(facility_summary.head())


In [ ]:
def plot_facility_amounts(cp):
    d = facility_summary[facility_summary["RMPM_ID"]==cp].sort_values("DATE_OF_COMMITTEE")
    plt.figure(figsize=(15,6))
    if "TOTAL_FINAL_TAKE" in d: plt.plot(d["DATE_OF_COMMITTEE"], d["TOTAL_FINAL_TAKE"], marker="o", label="Final Take")
    if "TOTAL_OUTSTANDING" in d: plt.plot(d["DATE_OF_COMMITTEE"], d["TOTAL_OUTSTANDING"], marker="o", label="Outstanding")
    plt.title(f"Facilities exposure - {cp}"); plt.legend(); plt.grid(alpha=.3)
    plt.xticks(rotation=45); plt.tight_layout(); plt.show()

for cp in TARGET_RMPM_IDS:
    plot_facility_amounts(cp)


In [ ]:
facility_change_df = facility_changes(facility)
display(facility_change_df)
facility.to_pickle(OUTPUT_DIR/"02_facility_enriched.pkl")
facility_summary.to_csv(OUTPUT_DIR/"facility_summary.csv", index=False)
facility_change_df.to_csv(OUTPUT_DIR/"facility_changes.csv", index=False)


## 7. Covenants / Assets


In [ ]:
covenant = pd.read_pickle(OUTPUT_DIR / "01_clean_covenant.pkl")
asset = pd.read_pickle(OUTPUT_DIR / "01_clean_asset.pkl")


In [ ]:
status_col = find_column(covenant, "STATUS")
type_col = find_column(covenant, "TYPE")
if status_col:
    display(covenant[status_col].value_counts(dropna=False).to_frame("count"))
    covenant["IS_BREACH"] = covenant[status_col].astype(str).str.lower().str.contains("breach", na=False)
    display(covenant[covenant["IS_BREACH"]])


In [ ]:
date_candidates = [c for c in ["PERFORMED_ON","RECEIPT_DATE","DUE_DATE","REPORTING_DATE","DATE_OF_COMMITTEE"] if c in covenant.columns]
if date_candidates:
    covenant["COVENANT_EVENT_DATE"] = covenant[date_candidates].bfill(axis=1).iloc[:,0]


In [ ]:
display(asset.groupby("RMPM_ID").size().rename("asset_rows").reset_index())
numeric = asset.select_dtypes(include=np.number).columns.tolist()
print("Numeric asset columns:", numeric)
if numeric:
    display(asset.groupby("RMPM_ID")[numeric].describe())
covenant.to_pickle(OUTPUT_DIR/"02_covenant_enriched.pkl")


## 8. Events / Timeline


In [ ]:
rmpm = pd.read_pickle(OUTPUT_DIR/"02_rmpm_enriched.pkl")
legal_entity = pd.read_pickle(OUTPUT_DIR/"01_clean_legal_entity.pkl")
facility_changes_df = pd.read_csv(OUTPUT_DIR/"facility_changes.csv")
covenant = pd.read_pickle(OUTPUT_DIR/"02_covenant_enriched.pkl")


In [ ]:
events = []

for cp, group in rmpm.groupby("RMPM_ID"):
    group = group.sort_values("SITUATION_DATE")
    for col, label in [("PD_CLEAN","PD change"),("GRR_SU","GRR change"),("RATING_NUM","Rating change")]:
        if col not in group.columns: continue
        previous = group[col].shift(1)
        mask = group[col].notna() & previous.notna() & (group[col] != previous)
        for idx in group[mask].index:
            events.append({
                "RMPM_ID":cp,"DATE":group.loc[idx,"SITUATION_DATE"],"SOURCE":"RMPM",
                "EVENT_TYPE":label,"OLD_VALUE":previous.loc[idx],"NEW_VALUE":group.loc[idx,col],
                "FACILITY_UNIQUE_ID":pd.NA
            })


In [ ]:
for col, on, off in [
    (find_column(legal_entity,"WATCHLIST_INDICATOR"),"Watchlist entry","Watchlist exit"),
    (find_column(legal_entity,"UTP_TEST"),"UTP entry","UTP exit"),
    (find_column(legal_entity,"CONFIRMED_UTP"),"Confirmed UTP","Confirmed UTP exit"),
    (find_column(legal_entity,"NPE"),"NPE entry","NPE exit"),
    (find_column(legal_entity,"Close to Default"),"Close to Default","Close to Default exit"),
]:
    if col:
        events += extract_boolean_transitions(legal_entity,col,"DATE_OF_COMMITTEE",on,off)


In [ ]:
if not facility_changes_df.empty:
    for _, row in facility_changes_df.iterrows():
        events.append({
            "RMPM_ID":row["RMPM_ID"],"DATE":row["DATE"],"SOURCE":"FACILITY",
            "EVENT_TYPE":f"Facility - {row['VARIABLE']} change",
            "OLD_VALUE":row["OLD_VALUE"],"NEW_VALUE":row["NEW_VALUE"],
            "FACILITY_UNIQUE_ID":row.get("FACILITY_UNIQUE_ID",pd.NA)
        })

status_col = find_column(covenant,"STATUS")
type_col = find_column(covenant,"TYPE")
if status_col and "COVENANT_EVENT_DATE" in covenant.columns:
    for _, row in covenant.iterrows():
        status = str(row.get(status_col,""))
        if any(k in status.lower() for k in ["breach","waiv","perform"]):
            events.append({
                "RMPM_ID":row.get("RMPM_ID"),"DATE":row.get("COVENANT_EVENT_DATE"),
                "SOURCE":"COVENANT","EVENT_TYPE":f"Covenant {status}",
                "OLD_VALUE":pd.NA,"NEW_VALUE":row.get(type_col,pd.NA) if type_col else pd.NA,
                "FACILITY_UNIQUE_ID":row.get("FACILITY_UNIQUE_ID",pd.NA)
            })

events = pd.DataFrame(events)
events["DATE"] = pd.to_datetime(events["DATE"],errors="coerce")
events = events.dropna(subset=["RMPM_ID","DATE"]).sort_values(["RMPM_ID","DATE"]).reset_index(drop=True)
display(events)
events.to_csv(OUTPUT_DIR/"events.csv", index=False)
events.to_pickle(OUTPUT_DIR/"events.pkl")


In [ ]:
def plot_event_timeline(cp):
    d = events[events["RMPM_ID"]==cp].copy()
    if d.empty: return
    labels = d["EVENT_TYPE"].dropna().unique().tolist()
    ymap = {e:i for i,e in enumerate(labels)}
    d["Y"] = d["EVENT_TYPE"].map(ymap)
    plt.figure(figsize=(16,max(6,len(labels)*.45)))
    plt.scatter(d["DATE"],d["Y"],s=70)
    plt.yticks(range(len(labels)),labels)
    plt.title(f"Event timeline - {cp}")
    plt.grid(axis="x",alpha=.3); plt.xticks(rotation=45); plt.tight_layout(); plt.show()

for cp in TARGET_RMPM_IDS:
    plot_event_timeline(cp)


## 9. Export final


In [ ]:
rmpm = pd.read_pickle(OUTPUT_DIR/"02_rmpm_enriched.pkl")
legal_entity = pd.read_pickle(OUTPUT_DIR/"01_clean_legal_entity.pkl")
facility = pd.read_pickle(OUTPUT_DIR/"02_facility_enriched.pkl")
covenant = pd.read_pickle(OUTPUT_DIR/"02_covenant_enriched.pkl")
asset = pd.read_pickle(OUTPUT_DIR/"01_clean_asset.pkl")
coverage = pd.read_csv(OUTPUT_DIR/"coverage.csv")
events = pd.read_pickle(OUTPUT_DIR/"events.pkl")


In [ ]:
out = OUTPUT_DIR/"structured_data_cleaned.xlsx"
with pd.ExcelWriter(out, engine="openpyxl") as writer:
    rmpm.to_excel(writer,sheet_name="RMPM",index=False)
    legal_entity.to_excel(writer,sheet_name="Legal_Entity",index=False)
    facility.to_excel(writer,sheet_name="Facility",index=False)
    covenant.to_excel(writer,sheet_name="Covenant",index=False)
    asset.to_excel(writer,sheet_name="Asset",index=False)
    coverage.to_excel(writer,sheet_name="Coverage",index=False)
    events.to_excel(writer,sheet_name="Events",index=False)
print(out)
